# BiCyc Multi-Adapter — Huấn luyện trên Google Colab

Chạy pipeline **KeepLoRA + BiCyc + PFD routing** (Hướng 1) trên GPU Colab thay vì máy local:
- GPU miễn phí: **T4 16GB** (khuyên dùng, hỗ trợ fp16 TensorCore).
- Bật **AMP fp16** + TF32 để tăng tốc ~1.5–2x.
- CIFAR-100 tự tải lần chạy đầu.
- Tuỳ chọn lưu dữ liệu/kết quả vào **Google Drive** để không mất khi phiên reset.

**Chuẩn bị code — chọn 1 trong 3 cách:**
- **Cách A (khuyên dùng):** điền `REPO_URL` (URL GitHub) ở cell dưới — Colab có Internet sẵn.
- **Cách B:** upload repo lên Drive, mount Drive rồi điền `DRIVE_REPO`.
- **Cách C:** kéo-thả file zip repo vào khung Files của Colab và giải nén tại `/content`.

**Chọn GPU:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# 1) Kiểm tra GPU
!nvidia-smi -L || echo "CANH BAO: chua bat GPU! Runtime -> Change runtime type -> T4 GPU"
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# 2) (Tuỳ chọn) Mount Google Drive để giữ data + kết quả qua các phiên
USE_DRIVE = True   # đặt False nếu chỉ chạy thử nhanh
DRIVE_MOUNTED = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNTED = True
print("Drive:", DRIVE_MOUNTED)

In [ ]:
# 3) Lấy code về /content/repo (Cách A: git clone | Cách B: Drive)
import shutil
from pathlib import Path

REPO_URL = ""    # Cách A: vd "https://github.com/<user>/BiCyc_MultiAdapter.git"
DRIVE_REPO = ""  # Cách B: vd "/content/drive/MyDrive/BiCyc_MultiAdapter"

WORK_REPO = Path("/content/repo")
candidates = [Path(p) for p in [DRIVE_REPO] if p]
repo = next((c for c in candidates if (c / "pyproject.toml").exists()), None)
if repo is None and REPO_URL:
    import subprocess
    if WORK_REPO.exists():
        shutil.rmtree(WORK_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK_REPO)], check=True)
    repo = WORK_REPO
assert repo is not None, "Dien REPO_URL hoac DRIVE_REPO o cell tren (hoac tai zip repo vao /content)."
if repo.resolve() != WORK_REPO.resolve():
    if WORK_REPO.exists():
        shutil.rmtree(WORK_REPO)
    shutil.copytree(repo, WORK_REPO, ignore=shutil.ignore_patterns(".git", "__pycache__", ".venv"))
print("Repo san sang:", WORK_REPO)

In [ ]:
# 4) Cài dependencies (không dùng editable install vì Colab runtime có thể là Python 3.13)
import os
import sys

repo_src = str(WORK_REPO / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = repo_src + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
print("Python:", sys.version)
if sys.version_info >= (3, 13):
    raise SystemExit(
        "RUNTIME KHONG HO TRO: Colab dang chay Python %d.%d.\n"
        "requirements/base.txt pin numpy==1.26.4 / scipy==1.14.1 / scikit-learn==1.5.2 /"
        " pandas==2.2.3 (khong co wheel cho 3.13)\nnên cai dat se that bai.\n"
        "-> Runtime > Change runtime type > Python 3.11 hoac 3.12 (T4 GPU) roi chay lai cell nay."
        % sys.version_info[:2]
    )
%pip install -q -r {WORK_REPO}/requirements/base.txt
sys.path.insert(0, repo_src)
import bicyc_multiadapter
print("bicyc_multiadapter", bicyc_multiadapter.__version__)

In [ ]:
# 3b) Chuẩn bị dữ liệu CIFAR-100 — lưu thẳng vào Drive nên KHÔNG bao giờ phải tải lại
import os
import subprocess
import sys
import tarfile
from pathlib import Path

repo_src = str(WORK_REPO / "src")
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    repo_src + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
)

_base = "/content/drive/MyDrive/bicyc" if DRIVE_MOUNTED else "/content"
DATA_ROOT = Path(f"{_base}/data/cifar100")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

extracted_dir = DATA_ROOT / "cifar-100-python"

if extracted_dir.is_dir() and any(extracted_dir.iterdir()):
    print(f"✅ Da co san du lieu tren Drive: {extracted_dir}")
else:
    tar_candidates = list(DATA_ROOT.glob("*.tar*")) + list(DATA_ROOT.glob("*.tgz"))
    if tar_candidates:
        tar_path = tar_candidates[0]
        print(f"📦 Tim thay file nen: {tar_path.name}. Dang giai nen...")
        with tarfile.open(tar_path, "r:*") as tar:
            tar.extractall(path=DATA_ROOT)
        print(f"✅ Giai nen thanh cong vao: {DATA_ROOT}")
    else:
        print("⬇️ Tai CIFAR-100 lan dau (~170MB)...")
        env = dict(os.environ, PYTHONPATH=os.environ["PYTHONPATH"])
        subprocess.run(
            [sys.executable, "-m", "bicyc_multiadapter.data.prepare", "--root", str(DATA_ROOT)],
            cwd=str(WORK_REPO),
            env=env,
            check=True,
        )
        if not extracted_dir.is_dir():
            for tar_path in DATA_ROOT.glob("*.tar*"):
                print(f"📦 Dang giai nen {tar_path.name}...")
                with tarfile.open(tar_path, "r:*") as tar:
                    tar.extractall(path=DATA_ROOT)

print("DATA_ROOT =", str(DATA_ROOT))

## Cấu hình thí nghiệm

| Preset | Khi nào dùng |
| --- | --- |
| `keeplora_bicyc` | Đề xuất đầy đủ (routed multi-adapter + BiCyc + adaptive gate) — cần GPU **≥ 24GB** (batch 128) |
| `keeplora_bicyc_8gb` | **T4/P100 16GB** — batch 32 + AMP bật sẵn (mặc định notebook) |
| `keeplora_original` | Baseline KeepLoRA nguyên gốc (merge sau task) |

**Chọn GPU (bắt buộc đọc):**
- **T4 / P100 / RTX 16GB** → dùng `keeplora_bicyc_8gb`, batch 32 (hạ xuống 16 nếu OOM).
- **A100 / V100 / L4 24GB+** → mới chọn `keeplora_bicyc` (batch 128).
- Chọn sai preset/batch sẽ **OOM ngay epoch đầu** — luôn chạy `SMOKE_TEST = True` trước.

> ⚠️ ViT-B/16 @224 × 20 epochs × 10 tasks **rất lâu** (hàng giờ đến >12h). Quy trình chuẩn:
> 1. `SMOKE_TEST = True` (cell 5): 10 tasks × 1 epoch, batch 16 → peak VRAM ~4GB, **không thể OOM**
>    trên T4 — xác minh pipeline chạy đủ 10 tasks và tạo đủ `run.log`/`run_meta.json`/`train_log.csv`.
> 2. Nếu ổn, tắt `SMOKE_TEST` và chạy thật.
> Colab free giới hạn phiên ~4–6h và có thể ngắt bất kỳ lúc nào — dùng `USE_DRIVE = True` để dữ liệu/kết
> quả không mất; nếu dở dang, tải `checkpoint_last.pt` về và đánh giá local bằng
> `python -m bicyc_multiadapter.evaluate`.


In [ ]:
# 5) Tham số run — sửa ở đây thay vì sửa yaml
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM

# =====================================================================
#  SMOKE TEST: xác minh pipeline chạy đủ 10 tasks KHÔNG bị OOM trên T4.
#  True  -> ghi đè cấu hình bên dưới bằng bộ thông số tiết kiệm VRAM
#           (10 tasks x 1 epoch, batch 16, AMP, cache nhỏ) ~20-40 phút.
#           Kết quả KHÔNG có ý nghĩa thống kê — chỉ kiểm tra pipeline.
#  False -> chạy cấu hình thật bên dưới (nhiều giờ).
# =====================================================================
SMOKE_TEST = True

# ---------------- cấu hình chạy thật (SMOKE_TEST = False) ----------------
EXPERIMENT = "keeplora_bicyc_8gb"  # T4/P100 16GB: DÙNG *_8gb. GPU >=24GB (A100/L4 24GB): keeplora_bicyc
SEED = 2024
EPOCHS_PER_TASK = 20
BATCH_SIZE = 32                    # T4/P100: 32. Giảm xuống 16 nếu OOM. Chỉ batch 128 khi GPU >=24GB
USE_AMP = True                     # fp16 mixed precision (hiệu quả nhất trên T4)
NUM_WORKERS = 2                    # Colab có 2 vCPU
ACTIVATION_CACHE_ROWS = 1536       # RAM CPU (không phải VRAM) cho SVD cuối task; 4096 nếu RAM >=16GB
CHECKPOINT_EVERY_EPOCHS = 1        # snapshot mỗi N epoch để resume khi Colab ngắt (~350MB, ghi đè)

# ---------------- cấu hình smoke test (SMOKE_TEST = True) ----------------
# Dự toán VRAM trên T4 16GB (ViT-B/16 @224, AMP fp16):
#   backbone + teacher snapshot ~0.9GB
#   activations (batch 16)      ~2.5GB
#   optimizer + BiCyc + head    ~0.3GB
#   => peak ~4GB: an toàn tuyệt đối. (batch 32 -> peak ~6GB, vẫn OK trên T4)
if SMOKE_TEST:
    EXPERIMENT = "keeplora_bicyc_8gb"
    EPOCHS_PER_TASK = 1
    BATCH_SIZE = 16
    ACTIVATION_CACHE_ROWS = 1024
    CHECKPOINT_EVERY_EPOCHS = 0    # 1 epoch/task: boundary checkpoint sau mỗi task là đủ
    print(">>> SMOKE TEST: 10 tasks x 1 epoch | batch 16 | AMP on | peak VRAM ~4GB")

# True: ghi checkpoint truc tiep len Google Drive (~350MB, cham 20-60s/lan, de bi
# rate-limit) nhung giu duoc qua cac phien de resume. False (mac dinh): ghi o dia
# local /content (nhanh) va zip+copy sang Drive o cell cuoi.
OUTPUTS_ON_DRIVE = False
_base = "/content/drive/MyDrive/bicyc" if DRIVE_MOUNTED else "/content"
DATA_ROOT = f"{_base}/data/cifar100"      # data tai 1 lan, giu tren Drive khong tai lai
if OUTPUTS_ON_DRIVE and DRIVE_MOUNTED:
    OUT_DIR = f"{_base}/outputs/{EXPERIMENT}/seed_{SEED}"   # resume duoc sau khi mat session
else:
    OUT_DIR = f"/content/outputs/{EXPERIMENT}/seed_{SEED}"  # nhanh; copy sang Drive o cell cuoi

overrides = [
    f"experiment={EXPERIMENT}",
    f"experiment.seed={SEED}",
    f"experiment.data.root={DATA_ROOT}",
    f"experiment.data.num_workers={NUM_WORKERS}",
    f"output_dir={OUT_DIR}",
    f"experiment.activation_cache_rows={ACTIVATION_CACHE_ROWS}",
    f"experiment.train.amp={str(USE_AMP).lower()}",
    f"experiment.checkpoint_every_epochs={CHECKPOINT_EVERY_EPOCHS}",
]
if EPOCHS_PER_TASK is not None:
    overrides.append(f"experiment.train.epochs_per_task={EPOCHS_PER_TASK}")
if BATCH_SIZE is not None:
    overrides.append(f"experiment.train.batch_size={BATCH_SIZE}")
print("Hydra overrides:\n  " + "\n  ".join(overrides))

In [ ]:
# 6) HUẤN LUYỆN — Gọn gàng, chỉ báo cáo khi hoàn thành từng Epoch / Task kèm ETA
import os
import subprocess
import sys

repo_src = str(WORK_REPO / "src")
env = dict(
    os.environ,
    PYTHONPATH=repo_src
    + (
        os.pathsep + os.environ.get("PYTHONPATH", "")
        if os.environ.get("PYTHONPATH")
        else ""
    ),
    PYTHONUNBUFFERED="1",
    TQDM_DISABLE="1",
    TF_CPP_MIN_LOG_LEVEL="3",
)

cmd = [sys.executable, "-u", "-m", "bicyc_multiadapter.train", *overrides]
print("$", " ".join(cmd), "\n" + "=" * 80)

process = subprocess.Popen(
    cmd,
    cwd=str(WORK_REPO),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(process.stdout.readline, ""):
        if any(
            x in line
            for x in [
                "Unable to register cu",
                "external/local_xla",
                "cpu_feature_guard",
            ]
        ):
            continue
        print(line, end="", flush=True)

    process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training that bai voi ma loi {return_code}")
    print("\n" + "=" * 80)
    print("🎉 HUAN LUYEN HOAN TAT THANH CONG!")
except KeyboardInterrupt:
    process.terminate()
    print("\n⚠️ Da tam dung huan luyen (Checkpoint da duoc tu dong luu lai).")

In [ ]:
# 7) (Tuỳ chọn) Theo dõi loss/metric trực tiếp bằng TensorBoard inline
%load_ext tensorboard
%tensorboard --logdir {OUT_DIR}/tensorboard

## Đánh giá & trực quan kết quả

In [ ]:
# 8) Đọc metrics, vẽ accuracy matrix và xuất báo cáo tổng kết dạng text (summary_report.txt)
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

out_dir = Path(OUT_DIR)
metrics_file = out_dir / "metrics.json"
history_file = out_dir / "history.jsonl"

if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
    summary = metrics.get("summary", {})
    raw_matrix = metrics.get("accuracy_matrix", [])
    num_tasks = len(raw_matrix)
    matrix = np.full((num_tasks, num_tasks), np.nan)
    for i, row in enumerate(raw_matrix):
        for j, val in enumerate(row):
            if j < num_tasks:
                matrix[i, j] = val
    
    # --- 1. Tạo file báo cáo tổng kết dạng text dễ đọc (summary_report.txt) ---
    report_lines = [
        "=" * 80,
        "                BICYC MULTI-ADAPTER EXPERIMENT REPORT (CIL 10 TASKS)",
        "=" * 80,
        f"Experiment : {EXPERIMENT}",
        f"Seed       : {SEED}",
        f"Output Dir : {OUT_DIR}",
        "-" * 80,
        "CIL CORE METRICS SUMMARY:",
        f"  * Last Average Accuracy (A_B)        : {summary.get('last_average', 0.0) * 100:.2f}% ({summary.get('last_average', 0.0):.4f})",
        f"  * Incremental Average Accuracy (A_bar): {summary.get('incremental_average', 0.0) * 100:.2f}% ({summary.get('incremental_average', 0.0):.4f})",
        f"  * Catastrophic Forgetting (F)         : {summary.get('forgetting', 0.0):.4f}",
        "-" * 80,
        "ACCURACY MATRIX (%):",
        "(Hang = sau khi hoc task i, Cot = do chinh xac tren task j)",
        "-" * 80,
    ]
    
    num_tasks = matrix.shape[0]
    header = "Task | " + " ".join(f"T{j:<5}" for j in range(num_tasks)) + " | Last_Avg"
    report_lines.append(header)
    report_lines.append("-" * len(header))
    
    for i in range(num_tasks):
        row_vals = []
        for j in range(num_tasks):
            if j <= i and not np.isnan(matrix[i, j]):
                row_vals.append(f"{matrix[i, j]*100:5.2f}%")
            else:
                row_vals.append("   -  ")
        valid_accs = [matrix[i, j] for j in range(i + 1) if not np.isnan(matrix[i, j])]
        row_avg = np.mean(valid_accs) * 100 if valid_accs else 0.0
        report_lines.append(f"T{i:02d}  | " + " ".join(row_vals) + f" | {row_avg:5.2f}%")
    
    report_lines.append("=" * 80)
    report_text = "\n".join(report_lines)
    
    report_path = out_dir / "summary_report.txt"
    report_path.write_text(report_text, encoding="utf-8")
    print(report_text)
    print(f"\nDa luu bao cao tong ket tai: {report_path}")
    
    # --- 2. Vẽ và lưu ảnh ma trận Accuracy ---
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f"{matrix[i, j]*100:.1f}%", ha="center", va="center", color="w", fontsize=8)
    ax.set_xlabel("Task da hoc (Seen Task)")
    ax.set_ylabel("Sau khi hoc Task (Evaluation Stage)")
    ax.set_title(f"{EXPERIMENT} (Seed {SEED})\nLast Avg: {summary.get('last_average', 0)*100:.2f}% | Forget: {summary.get('forgetting', 0):.4f}")
    fig.colorbar(im, ax=ax, label="Accuracy")
    plt.tight_layout()
    plt.savefig(out_dir / "accuracy_matrix.png", dpi=200)
    plt.show()


In [ ]:
# 9) Dọn dẹp checkpoint phụ và đóng gói kết quả nghiên cứu (chỉ giữ checkpoint_task_*.pt + logs quan trọng)
import os
import shutil
from pathlib import Path

out_dir = Path(OUT_DIR)

# Xóa các file checkpoint tạm/trùng lặp để tiết kiệm dung lượng (chỉ giữ lại checkpoint_task_*.pt)
redundant_files = ["checkpoint_boundary.pt", "checkpoint_live.pt", "checkpoint_last.pt"]
for rf in redundant_files:
    f_path = out_dir / rf
    if f_path.exists():
        f_path.unlink()

# Nén toàn bộ thư mục output thành results.zip
archive = shutil.make_archive("/content/results", "zip", out_dir)
print("=== DANH SACH CAC FILE DUOC LUU TRU TRONG RESULTS.ZIP ===")
for path in sorted(out_dir.iterdir()):
    if path.is_file():
        print(f"  [File] {path.name:30s} ({path.stat().st_size / 1024:,.1f} KB)")
    elif path.is_dir():
        print(f"  [Dir ] {path.name:30s}")

if DRIVE_MOUNTED and not str(Path(OUT_DIR)).startswith("/content/drive/"):
    dest = Path("/content/drive/MyDrive/bicyc_results")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, dest / Path(archive).name)
    print("\nDa sao luu results.zip vao Google Drive:", dest / Path(archive).name)

from google.colab import files
files.download(archive)


## Mẹo vận hành trên Colab

- **T4** hỗ trợ fp16 TensorCore → `USE_AMP = True` lợi ích lớn nhất; Colab Pro có A100/V100 hỗ trợ thêm bf16 (`experiment.train.amp_dtype=bfloat16`).
- **Smoke test không OOM**: `SMOKE_TEST = True` ở cell 5 chạy 10 tasks × 1 epoch, batch 16, AMP → peak
  VRAM ~4GB. Theo dõi dòng `GPU alloc=...GiB peak=...GiB` in sau mỗi task trong cell 6 (HUẤN LUYỆN);
  nếu peak vượt ~85% VRAM GPU, giảm `BATCH_SIZE` (32 → 16 → 8) trước khi bật chế độ thật.
- **`run.log` tự ghi đầy đủ cấu hình**: đầu run có bảng `RUN CONFIG` (experiment, seed, backbone,
  batch_size, epochs_per_task, lr/weight_decay, AMP, targets, alignment λ, cache rows...). Sau mỗi task:
  `acc= <row> | delta_cu=<thay đổi acc các task cũ> | last_avg inc_avg forget | time cum eta` — xem ngay
  sự suy giảm (forgetting) từng bước mà không cần mở file.
- Ablation nhanh bằng override trong `overrides`, ví dụ:
  - Tắt adaptive gate (λ_t ≡ 1): `experiment.alignment.adaptive_gate=false`
  - Baseline nguyên gốc: `experiment=keeplora_original`
  - Merge thay vì routed: `experiment.keeplora.merge_after_task=true`
  - Đổi seed: `experiment.seed=0` (chạy ≥ 3 seeds mỗi cấu hình khi báo cáo).
- Colab free **ngắt phiên bất định (~4–6h)**: luôn `USE_DRIVE = True`; data CIFAR-100 trong
  `Drive/bicyc/data/` được tái sử dụng nên không tải lại.
- **Tự động resume**: pipeline lưu `checkpoint_boundary.pt` (sau mỗi task) và `checkpoint_live.pt`
  (mỗi `CHECKPOINT_EVERY_EPOCHS` epoch hoặc khi interrupt). Colab ngắt phiên? Chỉ cần chạy lại cell
  huấn luyện với cùng `OUT_DIR` — training tiếp tục từ đúng epoch giữa task, kèm optimizer + RNG
  state. File checkpoint luôn ghi đè nên chỉ tốn dung lượng của 1 bản (~350MB với ViT-B).
- So sánh catastrophic forgetting: `history.jsonl` (mỗi task một dòng: per-task accuracy + tích lũy
  last/incremental/forgetting), TensorBoard scalar `eval/running_forgetting`, loss theo epoch trong
  `train_log.csv`.
- **Dataset không bao giờ tải lại**: CIFAR-100 được tải thẳng vào `Drive/bicyc/data/` ngay ở cell
  chuẩn bị dữ liệu; mọi phiên sau (kể cả resume giữa chừng) dùng lại tức thì.
- Muốn đánh giá lại không train: `python -m bicyc_multiadapter.evaluate experiment=<name>
  output_dir=<OUT_DIR> experiment.data.root=<DATA_ROOT>`.
